In [ ]:
#1
'''
Step 1: Normalization fix  ← الان در کد هست، باید تست بشه
'''

In [1]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
#5-band EEG (delta/theta/alpha/beta/gamma) -- see chat history
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # FIX: was E:\, change to match your actual drive

# NEW (per chat history -- QC-driven subject exclusion): VP004 excluded from training.
# Evidence (from eeg_preprocessing_qc.py, both the normal pipeline run and a subsequent strict
# re-preprocessing attempt with bad-segment annotation + exclusion before ICA fit):
#   - RAW: 65.1% of 2s epochs exceed the 150uV gross-amplitude threshold (vs <1-2% for healthy
#     subjects); 79.4% of the entire continuous recording was flagged as high-amplitude and
#     excluded from the ICA fit in the strict re-run.
#   - PREPROCESSED (normal pipeline): 13.3% of epochs still gross-amplitude, 24/28 channels
#     high-kurtosis.
#   - PREPROCESSED (strict re-run, ICA fit on only the cleanest 20.6% of the recording):
#     13.7% of epochs still gross-amplitude, 23/28 channels high-kurtosis -- i.e. NO measurable
#     improvement despite substantially restricting what ICA was allowed to learn from.
# The lack of improvement under stricter ICA fitting indicates the contamination is not a
# stationary, ICA-separable artifact source (e.g. a single stereotyped eye-blink component) but
# rather pervasive, non-stationary contamination (most likely continuous muscle tension/movement)
# that violates ICA's core assumption of temporally stable, statistically independent sources.
# This matches standard BCI/EEG QC protocols (e.g. FASTER, autoreject), which exclude subjects
# with >20-30% contaminated segments as unreliable for downstream analysis.
EXCLUDED_SUBJECTS = [4]  # NEW: VP004 -- see justification above
ALL_SUBJECTS = [s for s in range(1, 27) if s not in EXCLUDED_SUBJECTS]  # CHANGED: was list(range(1, 27))

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(RANDOM_SEED)

class MMapHybridDataset(Dataset): 

    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        import re, uuid, time  # NEW: uuid/time added for unique memmap filenames (see fix below)
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]

        # FIX (Windows/Jupyter file-lock hazard, per chat history): the previous version used a
        # FIXED filename per dataset_name (e.g. "t_mmap_train_set_e.npy"), then tried to
        # os.remove() + reopen it on every new instantiation. On Windows, a numpy memmap holds an
        # OS-level file lock for as long as the underlying mmap object (and the file handle behind
        # it) is alive. If you re-run a Jupyter cell that creates a NEW MMapHybridDataset before
        # the PREVIOUS one's memmap objects have been garbage-collected/cleanup()'d (e.g. you
        # re-ran the cell after an error, or forgot to call .cleanup()), os.remove() on that fixed
        # path raises PermissionError -- which the old code caught with a bare "except: pass" and
        # then continued on to open() the SAME (still-locked or stale) file. Depending on exact
        # timing this either crashes on the following open() call, or -- worse -- silently succeeds
        # while a stale/partially-overwritten file is used, meaning windows from a DIFFERENT
        # (possibly differently-sized/differently-filtered) previous run could silently leak into
        # the current training data with no error or warning at all.
        #
        # FIX: give every MMapHybridDataset instantiation its own UNIQUE filename (process id +
        # random hex + timestamp). This makes filename collisions between two live memmaps
        # (whether from the same process re-running a cell, or the old data still lingering)
        # essentially impossible, so there is no longer anything to silently overwrite. The
        # trade-off is that orphaned memmap files can accumulate on disk across repeated Jupyter
        # re-runs if cleanup() is never called (e.g. a cell erroring out before reaching
        # cleanup()) -- this fails SAFE (wasted disk space, loudly visible in the folder) instead
        # of failing SILENTLY (corrupted/stale training data), which is the property we actually
        # want here. Call .cleanup() when you're done with a dataset (already done in run_training
        # for both train_dataset/test_dataset), and periodically delete leftover
        # PROCESSED_DATA_PATH/t_mmap_*.npy files by hand if you interrupt runs often.
        unique_id = f"{os.getpid()}_{int(time.time())}_{uuid.uuid4().hex[:8]}"  # NEW
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_{unique_id}_e.npy")  # CHANGED: unique per instance
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_{unique_id}_n.npy")  # CHANGED: unique per instance

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                # With a unique filename this should never happen -- if it does, something is
                # unusual enough (e.g. a UUID collision, or manual file creation) that silently
                # continuing is not safe. FIX: loud warning instead of "except: pass".
                try:
                    os.remove(f_path)
                except PermissionError as e:  # FIX: no longer swallowed silently
                    raise PermissionError(
                        f"\n[MEMMAP FILE LOCK ERROR] Could not remove '{f_path}' (still locked by "
                        f"another process/handle) even though this filename should be unique to "
                        f"this run.\nThis means an old memmap from a PREVIOUS Jupyter cell run is "
                        f"still alive and holding the file open.\nFix: restart the Python/Jupyter "
                        f"kernel (this releases all OS-level file locks), or explicitly call "
                        f".cleanup() and `del` any previous MMapHybridDataset variables and run "
                        f"gc.collect() before re-running this cell.\nOriginal error: {e}"
                    ) from e
            with open(f_path, 'wb') as f:
                pass 
        
        # CHANGED (5-band EEG decomposition, per chat history): preprocessed EEG windows are now
        # (16, 16, 5, 600) grid tensors -- 5 standard frequency bands (delta/theta/alpha/beta/gamma),
        # each spatially mapped onto a 16x16 azimuthal-equidistant grid, stacked on a new axis right
        # after the spatial dims -- instead of the previous single-band (16, 16, 600) grid. This
        # memmap shape MUST match the .npy files actually saved on disk (VPxxx_eeg_windows.npy,
        # produced by the updated preprocessing script's extract_synchronized_windows(), which now
        # transposes eeg_slice_spatial as (16,16,5,600) -- see that script's comments), or
        # np.load()-ing them into this memmap slice will fail with a shape/broadcast error the
        # moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 5, 600))  # CHANGED: (N,16,16,600) -> (N,16,16,5,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # unchanged (fNIRS branch untouched)
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        n_files = len(self.valid_files)
        for file_idx, f_info in enumerate(self.valid_files):
            s, e = f_info['start_idx'], f_info['end_idx']
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            # progress print -- keeps Jupyter connection alive during long streaming
            # and shows exactly which subject is being written so crash location is visible
            print(f"  [{file_idx+1:02d}/{n_files}] {f_info['subject']} "
                  f"({e-s} windows, cumulative {e}/{self.total_windows})", flush=True)
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()

        # FIX (critical -- per chat history): the preprocessing script applied per-recording
        # z-score normalization (mean/std computed over the ENTIRE continuous recording, mixing
        # all 0/2/3-back trials together). This destroyed the between-class discriminative
        # information: all 3 class means became ~0 (measured SNR=0.0001, i.e. between-class
        # difference was 10,000x smaller than noise). A model trained on this data has essentially
        # no discriminative signal to learn from for EEG.
        # FIX: per-window z-score normalization -- each window is normalized independently,
        # so between-window (between-class) differences are preserved. This is the standard
        # approach in EEG literature (e.g. EEGNet, FBCSP, most BCI papers).
        eeg_mean = eeg_tensor.mean()
        eeg_std  = eeg_tensor.std() + 1e-8
        eeg_tensor = (eeg_tensor - eeg_mean) / eeg_std  # FIX: per-window, not per-recording

        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        if hasattr(self, "X_eeg"):
            del self.X_eeg
        if hasattr(self, "X_nirs"):
            del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try:
                    os.remove(f)
                except Exception as e:  # FIX: was a bare "except: pass" -- now visible, not silent
                    print(f"  WARNING: could not delete leftover memmap file '{f}': {e}\n"
                          f"           (harmless disk-space leak -- delete it by hand later, "
                          f"or restart the kernel to release the lock)")

    def __del__(self):
        # NEW: best-effort safety net for the case where a Jupyter cell errors out (or the user
        # simply forgets) before cleanup() is explicitly called -- when the dataset object is
        # eventually garbage-collected, try to release the memmap and remove its file. This is
        # NOT a substitute for calling cleanup() explicitly (Python's GC timing is not guaranteed,
        # especially in Jupyter where variables can be kept alive by output history/tracebacks),
        # but reduces how often stale memmap files pile up across repeated re-runs.
        try:
            self.cleanup()
        except Exception:
            pass  # __del__ must never raise -- if this fails, the file is just left for manual cleanup

# MODEL COMPONENTS (TSFNet)

def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    a_flat = a.reshape(-1)  # NEW
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # CHANGED (5-band EEG decomposition, per chat history): EEG input is no longer a single
        # broadband (16,16,600) grid (channel=1). It is now 5 spatially-mapped grids -- one per
        # standard frequency band (delta/theta/alpha/beta/gamma) -- stacked as 5 input "channels"
        # to the 3D convolution, analogous to a 5-channel image instead of a 1-channel (grayscale)
        # image. eeg_ch (the raw electrode count, 28) is STILL unrelated to this and kept only for
        # call-site backward compatibility; the actual in_channels below is fixed to 5 (band
        # count), not eeg_ch and not 1 anymore.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(5, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # CHANGED: 1 -> 5 (5 frequency bands)
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch (unchanged)
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # CHANGED (5-band EEG): fus_conv1 is fed by the SAME eeg_in tensor as eeg_conv1 (see
        # forward()), so it must also accept 5 input channels now instead of 1.
        self.fus_conv1 = nn.Conv3d(5, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # CHANGED: 1 -> 5 (5 frequency bands)
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        self._eeg_shape_printed = False  # FIX: guard for the one-time sanity print in forward()
        
    def forward(self, eeg, nirs, return_attention=False):  # PATCH: return_attention for xai_analysis.py
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        # CHANGED (5-band EEG decomposition, per chat history): eeg now arrives as
        # (batch, 16, 16, 5, 600) -- dims (batch, H, W, band, time) -- matching the preprocessing
        # script's saved window shape (16,16,5,600). Conv3d(in_channels=5, ...) expects
        # (batch, channels=5, depth=600, H=16, W=16), so permute maps:
        #   dim0(batch)->0, dim3(band)->1, dim4(time)->2, dim1(H)->3, dim2(W)->4
        # i.e. eeg.permute(0, 3, 4, 1, 2). (Previously, with a single broadband channel, this was
        # eeg.permute(0,3,1,2).unsqueeze(1) on a (batch,16,16,600) tensor -- the extra band axis
        # here replaces the unsqueeze(1) step.)
        eeg_in = eeg.permute(0, 3, 4, 1, 2)  # CHANGED: (batch,16,16,5,600) -> (batch,5,600,16,16)
        if not self._eeg_shape_printed:  # FIX: was printing on every single forward() call (every batch,
            print(f"[TSFNet.forward] eeg_in shape after permute: {tuple(eeg_in.shape)} "  # every epoch) --
                  f"(expected (batch,5,600,16,16))")  # now prints once, on the first call only.
            self._eeg_shape_printed = True  # FIX
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8)
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W')
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]

        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))
        fx2 = self.fus_drop2(fx2)

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W')
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w1 = torch.sigmoid(self.w1)  # sigmoid مطابق Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out

        if return_attention:  # PATCH: for xai_analysis.py -- returns Phi1/Phi2 attention maps
            return (final_out, eeg_out, fnirs_out, fusion_out,
                    eeg_feat_map, Phi2_bc_loss,
                    final_nirs, final_feat,
                    Phi1, Phi2)   # Phi1:(batch,1,H',W'), Phi2:(batch,1,H',W')
        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs


LAMBDA_EEG = 0.2     # Step 3 ablation: will increase to 0.4 after normalization fix is validated
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0

STAGE1_MAX_EPOCHS = 300   # paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # paper says patience 50
STAGE2_MAX_EPOCHS = 200   # paper Sec. 3.6 says max 200 for stage 2


def run_training(weight_decay, device, test_session=3):  # CROSS-SESSION: test_session = 1، 2 یا 3
    train_sessions = [s for s in [1, 2, 3] if s != test_session]  # CROSS-SESSION: دو session باقیمانده
    print("\n" + "#" * 70)
    print(f"# CROSS-SESSION: train on sessions {train_sessions}  |  test on session {test_session}")
    print("#" * 70)

    from torch.utils.data import SubsetRandomSampler

    train_dataset = MMapHybridDataset(ALL_SUBJECTS, "Train_Set",
                                       session_filter=train_sessions,
                                       min_windows_threshold=200)   # CROSS-SESSION
    test_dataset  = MMapHybridDataset(ALL_SUBJECTS, "Test_Set",
                                       session_filter=[test_session],
                                       min_windows_threshold=100)   # CROSS-SESSION

    all_trial_ids_global = []

    window_offset = 0
    for f_info in train_dataset.valid_files:
        subject = f_info['subject']
        sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
        trial_ids_path = os.path.join(sub_folder, f"{subject}_trial_ids.npy")
        
        trial_ids_full = np.load(trial_ids_path)  # همه ۲۷ trial
        
        if f_info['session_mask'] is not None:
            trial_ids_filtered = trial_ids_full[f_info['session_mask']]
        else:
            trial_ids_filtered = trial_ids_full
        
        n_windows_this_subject = f_info['end_idx'] - f_info['start_idx']
        assert len(trial_ids_filtered) == n_windows_this_subject, \
            f"{subject}: trial_ids mismatch ({len(trial_ids_filtered)} vs {n_windows_this_subject})"
        
        sub_idx = ALL_SUBJECTS.index(int(subject[2:]))
        global_trial_ids = sub_idx * 100 + trial_ids_filtered
        all_trial_ids_global.extend(global_trial_ids.tolist())
        window_offset += n_windows_this_subject
    
    all_trial_ids_global = np.array(all_trial_ids_global)
    assert len(all_trial_ids_global) == len(train_dataset), \
        f"trial_ids length mismatch: {len(all_trial_ids_global)} vs {len(train_dataset)}"
    
    unique_trials = np.unique(all_trial_ids_global)
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(unique_trials)
    
    split_trial = int(len(unique_trials) * 0.8)
    train_trials = set(unique_trials[:split_trial])
    val_trials   = set(unique_trials[split_trial:])
    
    s1_train_indices = [i for i, t in enumerate(all_trial_ids_global) if t in train_trials]
    s1_val_indices   = [i for i, t in enumerate(all_trial_ids_global) if t in val_trials]
    
    print(f"[Trial-level split] {len(unique_trials)} trials total | "
          f"{len(train_trials)} train trials ({len(s1_train_indices)} windows) | "
          f"{len(val_trials)} val trials ({len(s1_val_indices)} windows) | "
          f"ratio={len(s1_train_indices)/max(len(s1_val_indices),1):.1f}:1")
    print(f"[Trial-level split] Zero overlap guaranteed: "
          f"no window from a val trial appears in train")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)  # ablation step 1: keeping original dropout

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    optimizer = optim.Adam(model.parameters(), lr=0.001)  # ablation step 1: original settings

    _train_labels   = train_dataset.y
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float)
    _class_weights  = 1.0 / (_class_counts + 1e-6)
    _class_weights  = _class_weights / _class_weights.sum() * 3
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor)  # ablation step 1: no label smoothing

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{test_session}.pth')

    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of train sessions {train_sessions}), "
          f"val on {len(s1_val_indices)} windows (20%) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | window-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')
    stage1_final_train_loss    = float('inf')
    train_acc = 0.0
    history = {'epoch': [], 'train_acc': [], 'val_acc': [], 'train_loss': []}  # PATCH: for learning curves in xai_analysis.py

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)
        # PATCH: record epoch metrics for learning curves in xai_analysis.py
        history['epoch'].append(epoch)
        history['train_acc'].append(train_acc * 100)
        history['val_acc'].append(val_acc * 100)
        history['train_loss'].append(avg_train_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(ALL_SUBJECTS)} subjects, sessions {train_sessions}) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    optimizer2 = optim.Adam(model.parameters(), lr=0.001)  # ablation step 1: original settings

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    print(f"\n RUNNING FINAL TEST EVALUATION (test_session={test_session})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap_train_vs_test = (train_acc * 100) - test_acc
    overfit_gap_val_vs_test = (best_val_acc * 100) - test_acc

    print(f"\n[Reference, not primary] Stage-2 final train_acc - test_acc: {overfit_gap_train_vs_test:+.2f}pp "
          f"(NOTE: Stage 2 has no validation signal, so this is expected to be large "
          f"by design and does NOT indicate real overfitting)")
    print(f"Generalization gap (best Stage-1 val_acc - test_acc): {overfit_gap_val_vs_test:+.2f} percentage points "
          f"(this is the meaningful overfitting/generalization indicator, "
          f"since val is now trial-level split with no leakage)")

    results = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap_train_vs_test': overfit_gap_train_vs_test,
        'overfitting_gap_val_vs_test': overfit_gap_val_vs_test,
        'history': history,  # PATCH: Stage-1 learning curves for xai_analysis.py plot_learning_curves()
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_session{test_session}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results




In [ ]:
import torch, gc, os, glob

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# پاک‌سازی قبل از شروع
for f in glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy")):
    os.remove(f)
gc.collect()
torch.cuda.empty_cache()

result_s1 = run_training(weight_decay=0.0, device=device, test_session=1)
print(f"\nSession 1 done: {result_s1['test_accuracy_final']:.2f}%")


######################################################################
# CROSS-SESSION: train on sessions [2, 3]  |  test on session 1
######################################################################
Scanning Train_Set - 25 subjects (sessions [2, 3])...
VP001 verified: 649 windows
VP002 verified: 684 windows
VP003 verified: 684 windows
VP005 verified: 669 windows
VP006 verified: 684 windows
VP007 verified: 683 windows
VP008 verified: 681 windows
VP009 verified: 684 windows
VP010 verified: 679 windows
VP011 verified: 678 windows
VP012 verified: 684 windows
VP013 verified: 684 windows
VP014 verified: 677 windows
VP015 verified: 679 windows
VP016 verified: 684 windows
VP017 verified: 684 windows
VP018 verified: 684 windows
VP019 verified: 684 windows
VP020 verified: 684 windows
VP021 verified: 681 windows
VP022 verified: 674 windows
VP023 verified: 684 windows
VP024 verified: 684 windows
VP025 verified: 684 windows
VP026 verified: 680 windows
Streaming clean data to disk memory-map

In [ ]:
print(f"\nStep 1 (normalization fix only) - Session 1:")
print(f"  EEG-only:   {result_s1['ablation']['eeg_only']:.2f}%")
print(f"  fNIRS-only: {result_s1['ablation']['fnirs_only']:.2f}%")
print(f"  Fusion:     {result_s1['ablation']['fusion_only']:.2f}%")
print(f"  Final:      {result_s1['test_accuracy_final']:.2f}%")
print(f"  Kappa:      {result_s1['kappa_final']:.4f}")

In [ ]:
import time

# cooling break
print("Cooling 120s...")
time.sleep(120)

# پاک‌سازی
for f in glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy")):
    os.remove(f)
gc.collect()
torch.cuda.empty_cache()

result_s2 = run_training(weight_decay=0.0, device=device, test_session=2)
print(f"\nSession 2 done: {result_s2['test_accuracy_final']:.2f}%")

Cooling 120s...


In [ ]:
print("Cooling 120s...")
time.sleep(120)

for f in glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy")):
    os.remove(f)
gc.collect()
torch.cuda.empty_cache()

result_s3 = run_training(weight_decay=0.0, device=device, test_session=3)
print(f"\nSession 3 done: {result_s3['test_accuracy_final']:.2f}%")

In [ ]:
import numpy as np

results = [result_s1, result_s2, result_s3]
final_acc   = np.mean([r['test_accuracy_final'] for r in results])
final_kappa = np.mean([r['kappa_final'] for r in results])

print("="*50)
print(f"Session 1: {result_s1['test_accuracy_final']:.2f}%")
print(f"Session 2: {result_s2['test_accuracy_final']:.2f}%")
print(f"Session 3: {result_s3['test_accuracy_final']:.2f}%")
print(f"─────────────────────────────")
print(f"Final average accuracy: {final_acc:.2f}%")
print(f"Final average Kappa:    {final_kappa:.4f}")

In [ ]:
import torch, gc, os, glob, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def run_session(test_session, cooling_minutes=15):
    """یه session رو ران می‌کنه، بعدش cooling break می‌ذاره"""
    
    # پاک‌سازی قبل از شروع
    files = glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy"))
    if files:
        print(f"[CLEANUP] Deleting {len(files)} leftover memmap files...")
        for f in files:
            os.remove(f)
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"\n{'#'*60}")
    print(f"  Starting Session {test_session}")
    print(f"{'#'*60}\n")
    
    result = run_training(weight_decay=0.0, device=device, test_session=test_session)
    
    print(f"\n{'='*60}")
    print(f"Session {test_session} Results:")
    print(f"  EEG-only:   {result['ablation']['eeg_only']:.2f}%")
    print(f"  fNIRS-only: {result['ablation']['fnirs_only']:.2f}%")
    print(f"  Fusion:     {result['ablation']['fusion_only']:.2f}%")
    print(f"  Final:      {result['test_accuracy_final']:.2f}%")
    print(f"  Kappa:      {result['kappa_final']:.4f}")
    print(f"{'='*60}")
    
    return result


# ── اجرای هر ۳ session با cooling break ────────────────────
all_results = []

# Session 1
result_s1 = run_session(test_session=1)
all_results.append(result_s1)

# Cooling break
cooling = 15 * 60  # 15 دقیقه
print(f"\n[COOLING] Waiting {cooling//60} minutes before Session 2...")
for remaining in range(cooling, 0, -30):
    print(f"  {remaining//60}m {remaining%60}s remaining...", end='\r')
    time.sleep(30)
print("\n[COOLING] Done.")

# Session 2
result_s2 = run_session(test_session=2)
all_results.append(result_s2)

# Cooling break
print(f"\n[COOLING] Waiting {cooling//60} minutes before Session 3...")
for remaining in range(cooling, 0, -30):
    print(f"  {remaining//60}m {remaining%60}s remaining...", end='\r')
    time.sleep(30)
print("\n[COOLING] Done.")

# Session 3
result_s3 = run_session(test_session=3)
all_results.append(result_s3)

# ── نتیجه نهایی ────────────────────────────────────────────
import numpy as np
print(f"\n{'='*60}")
print(f"FINAL RESULTS (Step 1: Normalization Fix)")
print(f"{'='*60}")
print(f"{'Session':<12} {'EEG':>8} {'fNIRS':>8} {'Fusion':>8} {'Final':>8} {'Kappa':>8}")
for i, r in enumerate(all_results, 1):
    print(f"Session {i:<5} "
          f"{r['ablation']['eeg_only']:>7.2f}% "
          f"{r['ablation']['fnirs_only']:>7.2f}% "
          f"{r['ablation']['fusion_only']:>7.2f}% "
          f"{r['test_accuracy_final']:>7.2f}% "
          f"{r['kappa_final']:>8.4f}")

avg_acc   = np.mean([r['test_accuracy_final'] for r in all_results])
avg_kappa = np.mean([r['kappa_final'] for r in all_results])
print(f"{'─'*60}")
print(f"{'Average':<12} {'':>8} {'':>8} {'':>8} {avg_acc:>7.2f}% {avg_kappa:>8.4f}")

In [ ]:
import torch, gc, os, glob, time, numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def run_session(test_session):
    files = glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy"))
    if files:
        for f in files:
            os.remove(f)
    gc.collect()
    torch.cuda.empty_cache()
    result = run_training(weight_decay=0.0, device=device, test_session=test_session)
    print(f"\nSession {test_session}: Final={result['test_accuracy_final']:.2f}% | "
          f"EEG={result['ablation']['eeg_only']:.2f}% | "
          f"fNIRS={result['ablation']['fnirs_only']:.2f}% | "
          f"Fusion={result['ablation']['fusion_only']:.2f}%")
    return result

# Session 1 قبلاً ران شده -- نتیجه رو از فایل بخونید
import json
with open(os.path.join(PROCESSED_DATA_PATH, 'results_session1.json')) as f:
    result_s1 = json.load(f)
print(f"Session 1 (loaded): Final={result_s1['test_accuracy_final']:.2f}%")

# Cooling
cooling = 15 * 60
print(f"\n[COOLING] {cooling//60} minutes before Session 2...")
for remaining in range(cooling, 0, -30):
    print(f"  {remaining//60}m {remaining%60}s remaining...", end='\r')
    time.sleep(30)
print("\n[COOLING] Done.")

# Session 2
result_s2 = run_session(test_session=2)

# Cooling
print(f"\n[COOLING] {cooling//60} minutes before Session 3...")
for remaining in range(cooling, 0, -30):
    print(f"  {remaining//60}m {remaining%60}s remaining...", end='\r')
    time.sleep(30)
print("\n[COOLING] Done.")

# Session 3
result_s3 = run_session(test_session=3)

# نتیجه نهایی
all_results = [result_s1, result_s2, result_s3]
print(f"\n{'='*60}")
print(f"{'Session':<12} {'EEG':>8} {'fNIRS':>8} {'Fusion':>8} {'Final':>8}")
for i, r in enumerate(all_results, 1):
    print(f"Session {i:<5} "
          f"{r['ablation']['eeg_only']:>7.2f}% "
          f"{r['ablation']['fnirs_only']:>7.2f}% "
          f"{r['ablation']['fusion_only']:>7.2f}% "
          f"{r['test_accuracy_final']:>7.2f}%")
avg = np.mean([r['test_accuracy_final'] for r in all_results])
print(f"{'Average':<12} {'':>24} {avg:>7.2f}%")